# Hi-EF Phase 1 baseline smoke test

This notebook checks that B1 (clips I--II) and B2 (clips I--III) can train on the frozen source-folder split. It intentionally does not evaluate the test partition.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
FEATURES = Path('/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2')
subprocess.run(['git', '-C', str(REPO), 'pull', 'origin', 'experiments'], check=True)
assert (FEATURES / '01_00059.pt').exists()
print('Ready')

In [ ]:
common = [
    'python', str(REPO / 'experiments/train_baselines.py'),
    '--manifest', str(REPO / 'experiments/manifests/source_folder_split_seed42.csv'),
    '--features-dir', str(FEATURES),
    '--face-pooling', 'masked', '--seed', '42',
    '--epochs', '2', '--batch-size', '16', '--workers', '2',
    '--d-model', '256', '--temporal-layers', '1', '--inter-layers', '1',
    '--limit-train-batches', '10', '--limit-val-batches', '5'
]

In [ ]:
subprocess.run(common + [
    '--model', 'context',
    '--output-dir', '/kaggle/working/phase1_smoke/context_seed42'
], check=True)

In [ ]:
subprocess.run(common + [
    '--model', 'full',
    '--output-dir', '/kaggle/working/phase1_smoke/full_seed42'
], check=True)

In [ ]:
import json
for model in ('context_seed42', 'full_seed42'):
    path = Path('/kaggle/working/phase1_smoke') / model / 'metrics.json'
    metrics = json.loads(path.read_text())
    print(model, metrics['validation'])
    assert metrics['test'] is None
